# Lab | EDA bivariante: Amazon UK (parte II)

**Objetivo:** estudiar cómo se relacionan las variables de los productos de Amazon UK (categoría, best-seller, precio y valoración) para orientar decisiones de negocio.

**Datos:** [Amazon UK product dataset](https://www.kaggle.com/datasets/asaniczka/uk-optimal-product-price-prediction/) (Kaggle), el mismo del lab univariante.

> **Objectivo:** estudar como se relacionam as variáveis dos produtos da Amazon UK (categoria, best-seller, preço e avaliação) para orientar decisões de negócio.

## Antes de ejecutar

1. Dejar `amz_uk_price_prediction_dataset.csv` (descargado de Kaggle y descomprimido) **en esta carpeta**. Está en `.gitignore`.
2. Ejecutar todo (*Run All*). Con más de dos millones de filas, los gráficos de dispersión y de violín tardan algo más.
3. El informe final (sección 5) se escribe solo a partir de los resultados.

> **Antes de correr.** Deixar o `amz_uk_price_prediction_dataset.csv` **nesta pasta** (está no `.gitignore`) e correr tudo. Com mais de dois milhões de linhas, os gráficos de dispersão e de violino demoram um pouco mais. O relatório final escreve-se sozinho.

**Cómo funciona**

1. Importa las librerías de la lección `2.EDA_bivariate`: pandas, matplotlib, seaborn, y de `scipy.stats` el chi-cuadrado (`chi2_contingency`), la V de Cramér (`association`), la correlación de Pearson con su p-valor (`pearsonr`) y el gráfico QQ (`probplot`).
2. Lee el CSV y enseña su tamaño.

**Por qué así:** son las herramientas de las lecciones de EDA bivariante y de normalidad y correlación.

**Herramientas:** `import`, `pd.read_csv`, `shape`.

> **Como funciona (português)**
>
> 1. Importa as bibliotecas da lição `2.EDA_bivariate`: pandas, matplotlib, seaborn e, do `scipy.stats`, o qui-quadrado (`chi2_contingency`), o V de Cramér (`association`), a correlação de Pearson com o p-valor (`pearsonr`) e o gráfico QQ (`probplot`).
> 2. Lê o CSV e mostra o tamanho.
>
> **Porquê assim:** são as ferramentas das lições de EDA bivariada e de normalidade e correlação.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, pearsonr
from scipy.stats.contingency import association

df = pd.read_csv("amz_uk_price_prediction_dataset.csv")
print("Filas y columnas:", df.shape)
df.head()

**➜ Qué nos llevamos:** los datos cargados.

> **➜ O que tiramos:** os dados carregados.

## Parte 1. Best-sellers por categoría

> **Parte 1.** Best-sellers por categoria.

**Cómo funciona**

1. `pd.crosstab(df["category"], df["isBestSeller"])` cuenta, en cada categoría, cuántos productos son best-seller (`True`) y cuántos no (`False`).
2. Con `normalize="index"` cada fila suma 1: la columna `True` × 100 es el % de best-sellers de la categoría.
3. Se ordena de mayor a menor (`sort_values`) y se añade el número de productos de cada categoría, para no fiarse de un % calculado con pocos productos (en el informe solo se comparan categorías con 100 productos o más).

**Por qué así:** es la pista del enunciado: la proporción de best-sellers por categoría, ordenada. El número de productos al lado evita destacar categorías diminutas.

**Herramientas:** `pd.crosstab`, `normalize`, `sort_values`, `value_counts`, `pd.DataFrame`.

> **Como funciona (português)**
>
> 1. O `pd.crosstab(df["category"], df["isBestSeller"])` conta, em cada categoria, quantos produtos são best-seller (`True`) e quantos não (`False`).
> 2. Com `normalize="index"` cada linha soma 1: a coluna `True` × 100 é a % de best-sellers da categoria.
> 3. Ordena-se do maior para o menor (`sort_values`) e junta-se o número de produtos de cada categoria, para não confiar numa % calculada com poucos produtos.
>
> **Porquê assim:** é a pista do enunciado: a proporção de best-sellers por categoria, ordenada. O número de produtos ao lado evita destacar categorias minúsculas.

In [ ]:
tabla_best = pd.crosstab(df["category"], df["isBestSeller"])
pct_best = pd.crosstab(df["category"], df["isBestSeller"], normalize="index")[True] * 100

best_por_categoria = pd.DataFrame({
    "pct_best_seller": pct_best.round(2),
    "productos": df["category"].value_counts(),
}).sort_values("pct_best_seller", ascending=False)

print(f"Best-sellers en todo el catálogo: {df['isBestSeller'].mean() * 100:.2f} %")
best_por_categoria.head(10)

**➜ Qué nos llevamos:** las categorías donde ser best-seller es más habitual.

> **➜ O que tiramos:** as categorias onde ser best-seller é mais habitual.

**Contraste chi-cuadrado**

- **H0:** ser best-seller es independiente de la categoría.
- **H1:** la proporción de best-sellers depende de la categoría.
- α = 0,05.

> **H0:** ser best-seller é independente da categoria. **H1:** a proporção de best-sellers depende da categoria. α = 0,05.

**Cómo funciona**

1. `chi2_contingency(tabla_best)` compara los recuentos reales con los que habría si categoría y best-seller fueran independientes, y devuelve el estadístico y el p-valor.
2. `association(tabla_best, method="cramer")` da la V de Cramér: la fuerza de la relación, de 0 (ninguna) a 1 (total).

**Por qué así:** el chi-cuadrado dice si hay relación; la V de Cramér, si esa relación es fuerte. Con millones de filas casi todo sale significativo, así que la V de Cramér es la que dice si importa.

**Herramientas:** `chi2_contingency`, `association`, `if ... else`.

> **Como funciona (português)**
>
> 1. O `chi2_contingency(tabla_best)` compara as contagens reais com as que haveria se categoria e best-seller fossem independentes, e devolve a estatística e o p-valor.
> 2. O `association(tabla_best, method="cramer")` dá o V de Cramér: a força da relação, de 0 (nenhuma) a 1 (total).
>
> **Porquê assim:** o qui-quadrado diz se há relação; o V de Cramér, se essa relação é forte. Com milhões de linhas quase tudo sai significativo, por isso é o V de Cramér que diz se importa.

In [ ]:
chi2_stat, chi2_p, _, _ = chi2_contingency(tabla_best)
v_cramer = association(tabla_best, method="cramer")
print(f"Chi-cuadrado = {chi2_stat:,.0f}   p-valor = {chi2_p:.2e}   V de Cramér = {v_cramer:.3f}")
if chi2_p < 0.05:
    print("Rechazamos H0: la proporción de best-sellers depende de la categoría")
else:
    print("No podemos rechazar H0")

**➜ Qué nos llevamos:** si la categoría influye en ser best-seller y con qué fuerza.

> **➜ O que tiramos:** se a categoria influencia ser best-seller e com que força.

**Cómo funciona**

1. Toma las 10 categorías con más productos.
2. Con la tabla de proporciones (`normalize="index"`) dibuja una barra apilada por categoría con `plot(kind="bar", stacked=True)`, como en la lección.

**Por qué así:** con cientos de categorías la barra apilada no se lee; con las 10 más grandes y en %, se compara la parte de best-sellers de cada una.

**Herramientas:** `value_counts`, `head`, `isin`, `pd.crosstab`, `plot(kind="bar", stacked=True)`.

> **Como funciona (português)**
>
> 1. Pega nas 10 categorias com mais produtos.
> 2. Com a tabela de proporções desenha uma barra empilhada por categoria com `plot(kind="bar", stacked=True)`, como na lição.
>
> **Porquê assim:** com centenas de categorias a barra empilhada não se lê; com as 10 maiores e em %, compara-se a parte de best-sellers de cada uma.

In [ ]:
top_10_categorias = df["category"].value_counts().head(10).index
df_top_10 = df[df["category"].isin(top_10_categorias)]
apilada = pd.crosstab(df_top_10["category"], df_top_10["isBestSeller"], normalize="index") * 100

apilada.plot(kind="bar", stacked=True, figsize=(11, 5))
plt.title("% de best-sellers en las 10 categorías con más productos")
plt.ylabel("% de productos")
plt.xlabel("")
plt.show()

**➜ Qué nos llevamos:** una imagen de lo poco habitual que es ser best-seller incluso en las categorías grandes.

> **➜ O que tiramos:** uma imagem de como é pouco habitual ser best-seller mesmo nas categorias grandes.

## Parte 2. Precios y valoraciones por categoría

### Paso 0. Quitar los precios atípicos (método IQR)

> **Passo 0.** Tirar os preços atípicos (método IQR).

**Cómo funciona**

1. La función `limites_iqr` calcula Q1, Q3 y el IQR = Q3 − Q1, y devuelve los límites Q1 − 1,5 × IQR y Q3 + 1,5 × IQR (la función de Tukey de la lección).
2. Se quedan los productos con el precio dentro de los límites (`between`) en `df_sin_atipicos`.
3. Se enseña cuántos productos se quitan.

**Por qué así:** es lo que pide el enunciado: los precios extremos deforman medias y gráficos. Todo lo que sigue usa la tabla sin atípicos.

**Herramientas:** funciones, `quantile`, `between`, `len`.

> **Como funciona (português)**
>
> 1. A função `limites_iqr` calcula Q1, Q3 e o IQR = Q3 − Q1, e devolve os limites Q1 − 1,5 × IQR e Q3 + 1,5 × IQR (a função de Tukey da lição).
> 2. Ficam os produtos com o preço dentro dos limites (`between`) em `df_sin_atipicos`.
> 3. Mostra-se quantos produtos se tiram.
>
> **Porquê assim:** é o que pede o enunciado: os preços extremos deformam médias e gráficos. Tudo o que vem a seguir usa a tabela sem atípicos.

In [ ]:
def limites_iqr(serie):
    """Límites inferior y superior para detectar atípicos con el método IQR (Tukey)."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


limite_inf, limite_sup = limites_iqr(df["price"])
df_sin_atipicos = df[df["price"].between(limite_inf, limite_sup)]
print(f"Límites: de {limite_inf:.2f} a {limite_sup:.2f} £")
print(f"Productos quitados: {len(df) - len(df_sin_atipicos):,} ({(1 - len(df_sin_atipicos) / len(df)) * 100:.1f} %)")

**➜ Qué nos llevamos:** una tabla sin precios extremos para comparar categorías con justicia.

> **➜ O que tiramos:** uma tabela sem preços extremos para comparar categorias com justiça.

**Cómo funciona**

1. Violín del precio de las 20 categorías con más productos (`sns.violinplot`), como en la lección, con la categoría en el eje Y (`y="category"`): los violines quedan horizontales y los nombres largos se leen sin girarlos.
2. Mediana del precio de **todas** las categorías con `groupby` + `median`, ordenada de mayor a menor.

**Por qué así:** el violín enseña la forma completa del precio en cada categoría; la mediana por categoría responde a la pregunta sin depender de los 20 primeros.

**Herramientas:** `isin`, `sns.violinplot`, `groupby`, `median`, `sort_values`.

> **Como funciona (português)**
>
> 1. Violino do preço das 20 categorias com mais produtos (`sns.violinplot`), como na lição, com a categoria no eixo Y: os violinos ficam horizontais e os nomes compridos lêem-se sem os rodar.
> 2. Mediana do preço de **todas** as categorias com `groupby` + `median`, ordenada do maior para o menor.
>
> **Porquê assim:** o violino mostra a forma completa do preço em cada categoria; a mediana por categoria responde à pergunta sem depender das 20 primeiras.

In [ ]:
top_20_categorias = df_sin_atipicos["category"].value_counts().head(20).index
df_top_20 = df_sin_atipicos[df_sin_atipicos["category"].isin(top_20_categorias)]

plt.figure(figsize=(10, 10))
sns.violinplot(data=df_top_20, x="price", y="category")
plt.title("Precio en las 20 categorías con más productos (sin atípicos)")
plt.show()

mediana_precio = df_sin_atipicos.groupby("category")["price"].median().sort_values(ascending=False)
mediana_precio.head(5)

**➜ Qué nos llevamos:** la categoría con el precio mediano más alto.

> **➜ O que tiramos:** a categoria com o preço mediano mais alto.

**Cómo funciona**

1. Precio medio de las 10 categorías con más productos: se calcula con `groupby` + `mean` y se dibuja con `sns.barplot` en horizontal (categoría en el eje Y). Al darle un solo valor ya calculado por barra, seaborn no añade nada encima.
2. Precio medio de **todas** las categorías con `groupby` + `mean`, ordenado.

**Por qué así:** la media responde a «qué categoría tiene el precio medio más alto»; comparada con la mediana, dice si en esa categoría hay productos caros que tiran hacia arriba.

**Herramientas:** `sns.barplot`, `groupby`, `mean`, `sort_values`.

> **Como funciona (português)**
>
> 1. Preço médio das 10 categorias com mais produtos: calcula-se com `groupby` + `mean` e desenha-se com `sns.barplot` na horizontal (categoria no eixo Y). Ao receber um só valor já calculado por barra, o seaborn não acrescenta nada por cima.
> 2. Preço médio de **todas** as categorias com `groupby` + `mean`, ordenado.
>
> **Porquê assim:** a média responde a «que categoria tem o preço médio mais alto»; comparada com a mediana, diz se nessa categoria há produtos caros que puxam para cima.

In [ ]:
top_10_sin = df_sin_atipicos["category"].value_counts().head(10).index
media_top_10 = df_sin_atipicos[df_sin_atipicos["category"].isin(top_10_sin)].groupby("category")["price"].mean().sort_values(ascending=False)
plt.figure(figsize=(9, 5))
sns.barplot(x=media_top_10.values, y=media_top_10.index, color="steelblue")
plt.title("Precio medio en las 10 categorías con más productos (sin atípicos)")
plt.show()

media_precio = df_sin_atipicos.groupby("category")["price"].mean().sort_values(ascending=False)
media_precio.head(5)

**➜ Qué nos llevamos:** la categoría con el precio medio más alto.

> **➜ O que tiramos:** a categoria com o preço médio mais alto.

**Cómo funciona**

1. Diagramas de caja de las estrellas en las 10 categorías con más productos (`sns.boxplot`), en horizontal (categoría en el eje Y).
2. Mediana de las estrellas de **todas** las categorías con `groupby` + `median`, ordenada.

**Por qué así:** la caja compara mediana y dispersión de la valoración entre categorías de un vistazo.

**Herramientas:** `sns.boxplot`, `groupby`, `median`, `sort_values`.

> **Como funciona (português)**
>
> 1. Caixas de bigodes das estrelas nas 10 categorias com mais produtos (`sns.boxplot`), na horizontal (categoria no eixo Y).
> 2. Mediana das estrelas de **todas** as categorias com `groupby` + `median`, ordenada.
>
> **Porquê assim:** a caixa compara mediana e dispersão da avaliação entre categorias num relance.

In [ ]:
plt.figure(figsize=(9, 6))
sns.boxplot(data=df_sin_atipicos[df_sin_atipicos["category"].isin(top_10_sin)], x="stars", y="category")
plt.title("Estrellas en las 10 categorías con más productos")
plt.show()

mediana_estrellas = df_sin_atipicos.groupby("category")["stars"].median().sort_values(ascending=False)
mediana_estrellas.head(5)

**➜ Qué nos llevamos:** la categoría mejor valorada y si algunas categorías grandes tienen muchos productos sin valorar (mediana 0).

> **➜ O que tiramos:** a categoria mais bem avaliada e se algumas categorias grandes têm muitos produtos sem avaliação (mediana 0).

## Parte 3. Precio y valoración

> **Parte 3.** Preço e avaliação.

**Cómo funciona**

1. `pearsonr(precio, estrellas)` da la correlación de Pearson (relación lineal) y su p-valor (lección de normalidad y correlación).
2. `corr(..., method="spearman")` da la de Spearman, que mide si suben juntas aunque no sea en línea recta (lección `2.EDA_bivariate`).

**Por qué así:** el coeficiente dice la fuerza de la relación (de −1 a 1) y el p-valor si es significativa. Con tantos datos, una correlación minúscula puede ser significativa: hay que mirar las dos cosas.

**Herramientas:** `pearsonr`, `corr` con `method`, `if`.

> **Como funciona (português)**
>
> 1. O `pearsonr(precio, estrellas)` dá a correlação de Pearson (relação linear) e o p-valor.
> 2. O `corr(..., method="spearman")` dá a de Spearman, que mede se sobem juntas mesmo que não seja em linha recta.
>
> **Porquê assim:** o coeficiente diz a força da relação (de −1 a 1) e o p-valor se é significativa. Com tantos dados, uma correlação minúscula pode ser significativa: é preciso olhar para as duas coisas.

In [ ]:
r_pearson, p_pearson = pearsonr(df_sin_atipicos["price"], df_sin_atipicos["stars"])
r_spearman = df_sin_atipicos["price"].corr(df_sin_atipicos["stars"], method="spearman")
print(f"Pearson:  r = {r_pearson:.3f}   p-valor = {p_pearson:.2e}")
print(f"Spearman: r = {r_spearman:.3f}")

**➜ Qué nos llevamos:** si los productos más caros están mejor o peor valorados.

> **➜ O que tiramos:** se os produtos mais caros são mais bem ou mais mal avaliados.

**Cómo funciona**

1. Gráfico de dispersión de estrellas frente a precio (`sns.scatterplot`). Como las estrellas van de 0,1 en 0,1, los puntos forman columnas: se mira si las columnas de más estrellas llegan a precios más altos o más bajos.
2. Mapa de calor de las correlaciones entre todas las variables numéricas (`select_dtypes("number").corr()` y `sns.heatmap`), como en la lección.
3. Gráfico QQ del precio con `stats.probplot(..., plot=plt)` (lección de normalidad): si los puntos siguen la recta, el precio es normal.

**Por qué así:** son los tres gráficos que pide el enunciado; el QQ es la forma visual de comprobar la normalidad.

**Herramientas:** `sns.scatterplot`, `select_dtypes`, `corr`, `sns.heatmap`, `stats.probplot`.

> **Como funciona (português)**
>
> 1. Gráfico de dispersão de estrelas contra preço (`sns.scatterplot`). Como as estrelas vão de 0,1 em 0,1, os pontos formam colunas: vê-se se as colunas de mais estrelas chegam a preços mais altos ou mais baixos.
> 2. Mapa de calor das correlações entre todas as variáveis numéricas (`select_dtypes("number").corr()` e `sns.heatmap`), como na lição.
> 3. Gráfico QQ do preço com `stats.probplot(..., plot=plt)`: se os pontos seguirem a recta, o preço é normal.
>
> **Porquê assim:** são os três gráficos que o enunciado pede; o QQ é a forma visual de verificar a normalidade.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df_sin_atipicos, x="stars", y="price")
plt.title("Precio frente a estrellas")
plt.show()

correlaciones = df_sin_atipicos.select_dtypes("number").corr()
plt.figure(figsize=(7, 6))
sns.heatmap(correlaciones, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlaciones entre variables numéricas")
plt.show()

plt.figure(figsize=(6, 5))
stats.probplot(df_sin_atipicos["price"], plot=plt)
plt.title("Gráfico QQ del precio")
plt.show()

**➜ Qué nos llevamos:** si hay alguna relación visible entre precio y valoración y si el precio se aleja de la normal.

> **➜ O que tiramos:** se há alguma relação visível entre preço e avaliação e se o preço se afasta da normal.

## 4. Bonus: los mismos resultados sin quitar los atípicos

> **Bonus:** os mesmos resultados sem tirar os atípicos.

**Cómo funciona**

1. Repite los cálculos clave con la tabla completa `df`: categoría con mayor precio medio y mediano, y correlaciones precio–estrellas.
2. Los junta con los de la tabla sin atípicos en una sola tabla para compararlos.

**Por qué así:** la media y la correlación de Pearson son sensibles a los extremos; la mediana y Spearman, mucho menos. La comparación enseña cuánto cambian.

**Herramientas:** `groupby`, `mean`, `median`, `sort_values`, `index[0]`, `corr`, `pd.DataFrame`.

> **Como funciona (português)**
>
> 1. Repete os cálculos-chave com a tabela completa `df`: categoria com maior preço médio e mediano, e correlações preço–estrelas.
> 2. Junta-os com os da tabela sem atípicos numa só tabela para os comparar.
>
> **Porquê assim:** a média e a correlação de Pearson são sensíveis aos extremos; a mediana e Spearman, muito menos. A comparação mostra quanto mudam.

In [ ]:
media_todos = df.groupby("category")["price"].mean().sort_values(ascending=False)
mediana_todos = df.groupby("category")["price"].median().sort_values(ascending=False)

pd.DataFrame({
    "sin_atipicos": [media_precio.index[0], mediana_precio.index[0], round(r_pearson, 3), round(r_spearman, 3)],
    "con_atipicos": [media_todos.index[0], mediana_todos.index[0],
                     round(df["price"].corr(df["stars"]), 3), round(df["price"].corr(df["stars"], method="spearman"), 3)],
}, index=["mayor precio medio", "mayor precio mediano", "Pearson precio-estrellas", "Spearman precio-estrellas"])

**➜ Qué nos llevamos:** cuánto cambian las conclusiones si no se quitan los atípicos.

> **➜ O que tiramos:** quanto mudam as conclusões se não se tirarem os atípicos.

## 5. Informe para negocio

La celda siguiente escribe el informe con los resultados anteriores (f-strings y condiciones `if`), así siempre cuadra con los datos.

> **Relatório para o negócio.** A célula seguinte escreve o relatório com os resultados anteriores, por isso bate sempre certo com os dados.

**Cómo funciona**

1. Toma las cifras ya calculadas y las escribe en frases con f-strings.
2. Usa `if` para elegir la interpretación: por ejemplo, la fuerza de la V de Cramér (menos de 0,1 débil, menos de 0,3 moderada, más fuerte) y de la correlación.

**Por qué así:** un informe escrito a mano puede quedar desfasado si cambian los datos; generado desde los resultados, no.

**Herramientas:** f-strings, `if ... elif ... else`, `abs`, `print`.

> **Como funciona (português)**
>
> 1. Pega nos números já calculados e escreve-os em frases com f-strings.
> 2. Usa `if` para escolher a interpretação: por exemplo, a força do V de Cramér (menos de 0,1 fraca, menos de 0,3 moderada, mais forte) e da correlação.
>
> **Porquê assim:** um relatório escrito à mão pode ficar desactualizado se os dados mudarem; gerado a partir dos resultados, não.

In [ ]:
if v_cramer < 0.1:
    fuerza = "débil"
elif v_cramer < 0.3:
    fuerza = "moderada"
else:
    fuerza = "fuerte"

print("1. BEST-SELLERS")
print(f"   - Solo el {df['isBestSeller'].mean() * 100:.2f} % de los productos es best-seller.")
grandes = best_por_categoria[best_por_categoria["productos"] >= 100]
print(f"   - Entre las categorías con 100 productos o más, la que tiene más best-sellers en proporción es «{grandes.index[0]}» ({grandes['pct_best_seller'].iloc[0]:.1f} %, {grandes['productos'].iloc[0]:,} productos).")
if chi2_p < 0.05:
    print(f"   - Chi-cuadrado: p-valor {chi2_p:.1e}. La relación entre categoría y best-seller es significativa y {fuerza} (V de Cramér {v_cramer:.3f}).")
else:
    print(f"   - Chi-cuadrado: p-valor {chi2_p:.2f}. No hay pruebas de relación entre categoría y best-seller (V de Cramér {v_cramer:.3f}).")

print("\n2. PRECIOS Y VALORACIONES POR CATEGORÍA (sin atípicos)")
print(f"   - Se quitaron {len(df) - len(df_sin_atipicos):,} productos con precio por encima de {limite_sup:.2f} £ (el límite inferior, {limite_inf:.2f} £, no quita ninguno si es negativo).")
print(f"   - Mayor precio mediano: «{mediana_precio.index[0]}» ({mediana_precio.iloc[0]:.2f} £).")
print(f"   - Mayor precio medio: «{media_precio.index[0]}» ({media_precio.iloc[0]:.2f} £).")
print(f"   - Mayor valoración mediana: «{mediana_estrellas.index[0]}» ({mediana_estrellas.iloc[0]:.1f} estrellas).")

print("\n3. PRECIO Y VALORACIÓN")
if abs(r_pearson) < 0.1:
    print(f"   - Prácticamente no hay relación entre precio y estrellas (Pearson {r_pearson:.3f}, Spearman {r_spearman:.3f}): un producto más caro no está mejor ni peor valorado.")
else:
    print(f"   - Hay una relación {'positiva' if r_pearson > 0 else 'negativa'} entre precio y estrellas (Pearson {r_pearson:.3f}, Spearman {r_spearman:.3f}).")
print("   - El gráfico QQ muestra si el precio se aleja de la normal; con una cola larga a la derecha, conviene usar la mediana y Spearman.")

print("\n4. RECOMENDACIONES")
if fuerza == "débil":
    print("   - Ser best-seller depende poco de la categoría: el producto concreto importa más que el sector.")
else:
    print("   - La categoría influye en ser best-seller: conviene priorizar las categorías con más proporción de best-sellers.")
print("   - Para comparar precios entre categorías, usar la mediana: la media se mueve mucho con los productos caros.")

**➜ Qué nos llevamos:** el informe de negocio que pide el enunciado, siempre coherente con los datos.

> **➜ O que tiramos:** o relatório de negócio que o enunciado pede, sempre coerente com os dados.